# Architecture Used in This Notebook

```text
                 ┌──────────────────────┐
                 │       PDF Files      │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │    Text Extraction   │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │       Chunking       │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │      Embeddings      │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │      ChromaDB        │
                 │   Vector Database    │
                 └──────────┬───────────┘
                            ↑
                     Similarity Search
                            ↑
                       User Question
                            ↓
                 ┌──────────────────────┐
                 │   Retrieved Context  │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │  Prompt Construction │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │ Open-Source LLM      │
                 │ Hugging Face         │
                 └──────────┬───────────┘
                            ↓
                         Answer
```

The infrastructure is intentionally lightweight so the training audience can focus on **RAG concepts and implementation**.

# Step 1: Install Required Open-Source Libraries

We deliberately avoid Databricks-specific packages.

- `pypdf` → PDF extraction
- `langchain-text-splitters` → chunking
- `sentence-transformers` → embeddings
- `chromadb` → vector database
- `transformers` → open-source LLM inference
- `accelerate` → model/runtime support
- `pandas` → inspection and simple data handling
- `torch` → model inference

The first run may download models from Hugging Face. After caching, later runs are faster.

**Colab:** run the cell and continue.

**VS Code:** use a Python notebook kernel with internet access for the initial model download.

# Step 2: Imports and Configuration

## Models

### Embedding model
`sentence-transformers/all-MiniLM-L6-v2`

- small and fast,
- 384-dimensional embeddings,

### Generation model
`google/flan-t5-base`

- open model,
- relatively lightweight,
- works on CPU and GPU,

For stronger GPU-based generation, the model can later be replaced with a larger instruction-tuned model.

# Step 3: Document Input

Here we use:

```text
project/
├── rag_open_source_demo.ipynb
├── data/
│   ├── document1.pdf
│   ├── document2.pdf
│   └── ...
└── chroma_db/
```

# Step 4: Extract Text from Documents

The ingestion layer should:

1. Find files.
2. Read them.
3. Extract text.
4. Preserve metadata.

Useful metadata includes:

- document ID,
- source file,
- page number,
- section,
- version,
- timestamp,
- access level.

## Why metadata matters

Metadata enables:

- citations,
- filtering,
- access control,
- debugging,
- document-level retrieval,
- traceability.

For example:

```text
content = "The warranty is valid for 24 months..."
source = "accessibility_manual.pdf"
page = 13
```

is much more useful than storing only the text.

                DATA/
                  │
          ┌───────┴────────┐
          ▼                ▼
        PDFs              TXT
          │                │
      PdfReader       read_text()
          │                │
      page by page       whole file
          │                │
          └───────┬────────┘
                  ▼
             documents
                  │
                  ▼
        list of dictionaries
                  │
                  ▼
           Pandas DataFrame

# Document Parsing: Real-World Considerations

`pypdf` works well for text-based PDFs, but enterprise documents can be more complex.

| Document | Typical open-source approach |
|---|---|
| Text PDF | `pypdf`, PyMuPDF |
| Scanned PDF | OCR |
| DOCX | `python-docx` |
| HTML | BeautifulSoup |
| Markdown | Native parsing |
| PowerPoint | `python-pptx` |
| Excel | `openpyxl`, pandas |
| Images | OCR / vision models |

## OCR

A scanned PDF may contain images rather than an actual text layer.

```text
Scanned PDF
    ↓
OCR
    ↓
Extracted Text
    ↓
Chunking
    ↓
Embeddings
```

Common open-source OCR options include **Tesseract** and **PaddleOCR**.

### Key lesson

Document parsing quality directly affects downstream retrieval quality.

# Step 5: Chunk the Documents

Large documents are normally divided into smaller pieces called **chunks**.

Chunking helps with:

- retrieval relevance,
- focused embeddings,
- smaller prompts,
- lower irrelevant context,
- better context selection.

A good chunk should represent a meaningful unit of information rather than an arbitrary slice.

# Chunking Strategies

The reference notebook introduced several approaches. We retain them here without Databricks dependencies.

### 1. Recursive Character Chunking

Uses a hierarchy of separators and tries to preserve larger semantic units before falling back to smaller ones.

**Good general-purpose default.**

### 2. Token-Based Chunking

Splits according to tokens.

Useful because LLM context windows are measured in tokens.

**Best for:** precise context-window control.

### 3. Markdown-Aware Chunking

Uses document headings and structure.

**Best for:** technical documentation and structured knowledge bases.

### 4. Sentence-Based Chunking

Keeps sentence boundaries.

**Best for:** FAQs, policies and narrative content.

### 5. Semantic Chunking

Groups content according to semantic similarity instead of only fixed size.

**Best for:** advanced pipelines where preserving meaning is especially important.

## Chunk size trade-off

Too small:

- context can be incomplete,
- retrieval may become fragmented.

Too large:

- embeddings become less focused,
- irrelevant context increases,
- prompt size increases.

There is no universal chunk size. It should be evaluated on the target documents and questions.

## Inspect Chunk Quality

Always inspect chunks before indexing.

Ask:

- Is the chunk too small?
- Does it contain multiple unrelated topics?
- Was a sentence or table broken incorrectly?
- Is overlap sufficient?
- Can the chunk stand on its own?

A RAG system can fail **before retrieval begins** if chunking is poor.

## Chunking Comparison

| Method | Splits by | Strength | Typical use |
|---|---|---|---|
| Recursive character | Hierarchical separators | Robust baseline | General RAG |
| Token-based | Token count | Context-window aware | LLM systems |
| Markdown-aware | Headers/structure | Preserves hierarchy | Documentation |
| Sentence-based | Sentences | Linguistic coherence | FAQ/policy |
| Semantic | Meaning | Strong semantic coherence | Advanced RAG |

**Practical recommendation:** start with recursive chunking, evaluate it, then tune or move to more advanced strategies if the evaluation justifies it.

# Optional Chunking Experiments

The next cells show the reference notebook's other chunking approaches.

They are for **learning and comparison**. We continue using recursive chunks for the actual vector index.

In [7]:
%pip install -q tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 6: Embeddings

An embedding model converts text into a numerical vector.

```text
"How long is the warranty?"
             ↓
       Embedding Model
             ↓
[0.12, -0.04, 0.77, ...]
```

Semantically similar text tends to have nearby vectors.

## Why embeddings?

Keyword search can miss semantic matches.

Question:

> "How long can I return the product?"

Document:

> "Returns are accepted within 30 days."

Embeddings can capture the semantic relationship even when wording differs.

### Important distinction

An embedding model does **not** answer the question.

It represents text numerically so that we can compare it with other text.

# Embedding Model Choices

This demo uses:

`sentence-transformers/all-MiniLM-L6-v2`

Other families worth knowing:

- BGE
- E5
- multilingual Sentence Transformers
- other Hugging Face embedding models

## How to choose

Consider:

1. Retrieval benchmark quality
2. Language coverage
3. Vector dimension
4. Latency
5. Memory
6. Maximum input length
7. Licensing
8. Domain performance

### Important

Changing the embedding model generally means **re-embedding the corpus**. Vectors produced by different embedding spaces should not simply be mixed.

# Step 7: Vector Databases

A vector database stores embeddings and provides similarity search.

```text
Chunk
  ↓
Embedding
  ↓
Vector Database
  ↓
Similarity Search(query)
  ↓
Top-K Chunks
```

## Why not just use a Python list?

For tiny experiments, you can.

For larger systems, you need:

- efficient nearest-neighbor search,
- metadata filtering,
- persistence,
- concurrent access,
- indexing,
- scalable storage.

## Vector database landscape

| Technology | Good fit |
|---|---|
| **ChromaDB** | Local development and training |
| **FAISS** | High-performance local vector search |
| **Qdrant** | Production-oriented open-source deployments |
| **Weaviate** | Rich vector search and hybrid retrieval |
| **Milvus** | Large-scale distributed vector workloads |
| **pgvector** | Vector search inside PostgreSQL |
| **OpenSearch / Elasticsearch** | Keyword + vector hybrid search |
| **LanceDB** | Embedded/local applications |
| **Vespa** | Advanced search and ranking systems |

### Why ChromaDB here?

Chroma can run locally with a simple Python API and persistent storage, so we can teach vector databases without setting up a separate service.

# Vector Database vs Vector Index

A **vector database** is the storage and query system.

A **vector index** is the data structure that makes nearest-neighbor search efficient.

Common approximate nearest-neighbor approaches include:

- HNSW
- IVF
- Product Quantization
- Disk-based ANN approaches

These involve trade-offs between:

**speed ↔ memory/storage ↔ recall ↔ indexing cost**

For this training notebook, Chroma abstracts these implementation details.

# Step 8: Similarity Search

The user's question is also converted into an embedding.

```text
Question
   ↓
Question Embedding
   ↓
Compare with document vectors
   ↓
Nearest neighbors
   ↓
Top-K chunks
```

## Similarity metrics

Common metrics:

- cosine similarity,
- dot product,
- Euclidean distance.

With normalized embeddings, cosine similarity and dot product are closely related.

## What does top-k mean?

`k` is the number of chunks retrieved.

Small k:

- less context,
- faster,
- may miss evidence.

Large k:

- more coverage,
- more irrelevant information,
- larger prompt.

Top-k should be evaluated rather than chosen blindly.

# Understanding Retrieval Results

During development, inspect:

- retrieved chunks,
- ranking,
- similarity/distance,
- source documents,
- page numbers.

Ask:

> **Did the system retrieve the right information?**

If the answer is no, changing the LLM prompt may not help.

You may need to improve:

- parsing,
- chunking,
- embeddings,
- metadata filtering,
- query formulation,
- retrieval,
- reranking.

# Step 9: Retrieval Improvements

Basic vector search is only the beginning.

## Metadata filtering

Retrieve only documents satisfying conditions such as:

```text
department = "HR"
year = 2026
document_type = "policy"
```

Useful for access control and domain filtering.

## Hybrid Search

Combine:

- keyword/lexical search,
- semantic/vector search.

This helps when exact terms matter, such as:

- error codes,
- product codes,
- policy numbers,
- legal clauses.

## Reranking

Retrieve a larger candidate set, then reorder it with a reranker:

```text
Query
 ↓
Vector retrieval: top 20
 ↓
Reranker
 ↓
Best 5
 ↓
LLM
```

## Query rewriting

Transform a vague user question into a better retrieval query.

## Multi-query retrieval

Generate multiple related queries and merge the results.

These techniques are common in advanced production RAG.

# Step 10: Build Retrieved Context

Retrieved chunks become the evidence provided to the LLM.

A useful context format is:

```text
[Source: accessibility_manual.pdf, page 4]
Chunk text...

[Source: accessibility_manual.pdf, page 7]
Chunk text...
```

Including source metadata makes debugging and citations easier.

### Important principle

The goal is not:

> retrieve as much as possible.

The goal is:

> retrieve a small amount of highly relevant evidence.

# Step 11: Prompt Engineering for RAG

A grounded RAG prompt should clearly separate:

1. Instructions
2. Retrieved context
3. User question
4. Output requirements

A strong baseline instruction is:

> Answer using only the provided context. If the context does not contain the answer, say that the information is not available in the provided documents.

This reduces unsupported answers.

### Important limitation

Prompt instructions **reduce** hallucination but do not guarantee zero hallucination.

RAG quality depends on the entire pipeline.

# Step 12: Open-Source LLM

Here we use **Hugging Face Transformers** and run the model locally.

### Why a small model?

For a training notebook we prioritize:

- low setup effort,
- reproducibility,
- reasonable RAM/VRAM,
- simple code.

`google/flan-t5-base` is not the strongest modern model, but it is convenient for understanding the mechanics.

## Larger model options

On a GPU environment you can explore:

- Qwen instruction models
- Llama instruction models
- Mistral instruction models
- Gemma instruction models
- Phi instruction models

Consider:

- answer quality,
- context length,
- VRAM/RAM,
- latency,
- licensing,
- quantization,
- language coverage.

## Production model serving

Instead of loading the model inside the notebook, production systems often use:

- vLLM
- Hugging Face TGI
- Ollama
- llama.cpp

The RAG application then calls the model through an API.

This separates **retrieval infrastructure** from **model serving**.

# LLM Parameters in RAG

Important generation concepts:

### temperature
Controls randomness.

- low → deterministic/factual
- high → more diverse

### max_new_tokens
Maximum output length.

### top_p
Controls nucleus sampling.

### repetition_penalty
Discourages repeated output.

### do_sample
Controls whether sampling is used.

For a factual RAG assistant, start conservatively. The model's job is generally to transform retrieved evidence into a useful answer, not to invent creative content.

# Step 13: Complete RAG Function

We now combine all components:

```text
Question
   ↓
Embed question
   ↓
Vector search
   ↓
Top-K chunks
   ↓
Build context
   ↓
Build grounded prompt
   ↓
LLM
   ↓
Answer
```

This is the core RAG loop.

# Step 14: Test with Multiple Questions

Try:

1. Which accessibility features are supported?
2. What are the environment requirements?
3. How long is the warranty?
4. What security controls are recommended?
5. How can performance be optimized?
6. What authentication methods are supported?
7. What should an administrator check during a login failure?
8. What information is not present in the documents?

### Exercise

Before reading the final answer, inspect the retrieved chunks.

Ask:

> "Would the model have enough evidence to answer this question?"

# Step 15: Metadata and Filtering

Metadata becomes critical in enterprise RAG.

Imagine the knowledge base contains:

```text
department = HR
department = Finance
department = Legal
```

A user asks:

> "What is our leave policy?"

The retrieval layer may need to restrict the search to HR content.

Typical metadata:

- `document_id`
- `source`
- `page`
- `department`
- `document_type`
- `region`
- `language`
- `version`
- `created_at`
- `access_level`
- `tenant_id`

### Security warning

Metadata filtering is not a replacement for authorization.

Sensitive content must not reach the LLM unless the user is authorized to retrieve it.

# Step 16: RAG Evaluation

Do not evaluate RAG only by asking:

> "Does the answer look good?"

Evaluate retrieval and generation separately.

## Retrieval metrics

### Recall@K
Did the relevant chunk appear in the top K?

### Precision@K
How many retrieved chunks were relevant?

### MRR
How highly ranked was the first relevant result?

## Generation metrics

### Faithfulness / Groundedness
Is the answer supported by retrieved context?

### Answer relevance
Does it answer the question?

### Context relevance
Was the retrieved context useful?

### Citation correctness
Do cited sources support the answer?

Useful evaluation ecosystems include **Ragas**, **DeepEval**, **promptfoo**, and custom evaluation pipelines.

Key distinction:

```text
Retrieval quality ≠ Answer quality
```

# Step 17: RAG Failure Modes

## 1. Ingestion failure
The document was not extracted correctly.

## 2. Chunking failure
Relevant information was split across chunks.

## 3. Embedding failure
Semantic relationships are not represented well.

## 4. Retrieval failure
The correct chunk exists but is not retrieved.

## 5. Context failure
Correct chunks are retrieved but irrelevant chunks distract the model.

## 6. Prompt failure
The model is not adequately constrained.

## 7. Generation failure
The LLM misunderstands or ignores evidence.

### Debug from left to right

```text
Documents
  ↓
Extraction
  ↓
Chunks
  ↓
Embeddings
  ↓
Retrieval
  ↓
Context
  ↓
Prompt
  ↓
LLM
  ↓
Answer
```

Always identify **which layer failed** before changing the system.

# Step 18: Production Retrieval Architecture

A more advanced RAG system can look like:

```text
                       User Query
                           ↓
                    Query Rewriting
                           ↓
              ┌────────────┴────────────┐
              ↓                         ↓
        Keyword Search             Vector Search
              ↓                         ↓
              └────────────┬────────────┘
                           ↓
                    Candidate Chunks
                           ↓
                       Reranker
                           ↓
                    Top Relevant Chunks
                           ↓
                  Context Construction
                           ↓
                         LLM
                           ↓
                  Answer + Citations
```

This is why a production RAG system is much more than:

```python
vector_db.similarity_search(question)
```

# Step 19: RAG vs Fine-Tuning vs Long Context

| Approach | Best suited for |
|---|---|
| RAG | External/private/changing knowledge |
| Fine-tuning | Behavior/style/task specialization |
| Long context | Information that fits into one request |
| RAG + fine-tuning | Knowledge + specialized behavior |

Example: an HR assistant needs the latest company policies.

- **RAG:** retrieves current policies.
- **Fine-tuning:** can teach response style or a specialized task.
- **Long context:** can pass a large document directly if it fits.

RAG is attractive for changing knowledge because updating the knowledge source does not require retraining the LLM.

# Step 20: Production RAG — What Changes?

This notebook intentionally removes infrastructure complexity.

A production system usually adds:

### Ingestion
- document pipelines
- OCR
- parsers
- deduplication
- versioning
- incremental indexing

### Retrieval
- hybrid search
- filters
- reranking
- query rewriting
- access control

### Generation
- hosted/self-hosted LLM
- model routing
- streaming
- structured output
- guardrails

### Operations
- logging
- tracing
- evaluation
- latency monitoring
- cost monitoring
- user feedback

### Security
- authentication
- authorization
- tenant isolation
- PII handling
- prompt-injection defenses
- document-level permissions

# Step 21: Hands-On Exercises

### Exercise 1 — Chunking
Compare `chunk_size` values of 400, 800 and 1200.

### Exercise 2 — Overlap
Compare overlap values of 0, 50, 100 and 150.

### Exercise 3 — Metadata filtering
Add `department` metadata and retrieve only one department.

### Exercise 4 — Citations
Modify the final answer to include:

```text
Answer: ...

Sources:
1. document.pdf, page 4
2. document.pdf, page 7
```

### Exercise 5 — Not-found evaluation
Create questions whose answers are absent. Check whether the system refuses to invent an answer.

### Exercise 6 — Embedding comparison
Replace MiniLM with another embedding model and compare retrieval quality.

### Exercise 7 — Reranking
Retrieve top-10 candidates, rerank them, then pass the top-3 to the LLM.

### Exercise 8 — Evaluation dataset
Create:

```text
question
expected_answer
expected_source
```

and measure retrieval and answer quality.

# Final Takeaways

1. **RAG is a pipeline, not just an LLM prompt.**
2. Document extraction quality affects everything downstream.
3. Chunking determines what the retriever can find.
4. Embeddings represent semantic meaning.
5. Vector databases make similarity retrieval practical.
6. Metadata enables filtering, traceability and access control.
7. Top-K should be evaluated.
8. Reranking and hybrid search can improve retrieval.
9. The LLM should generate from retrieved evidence.
10. Retrieval and generation require separate evaluation.
11. ChromaDB is convenient for learning; production systems may use Qdrant, Weaviate, Milvus, pgvector, OpenSearch/Elasticsearch, or others.
12. Open-source components let us learn RAG without cloud-specific setup.

## Final mental model

```text
                    RAG
                     │
        ┌────────────┼────────────┐
        ↓            ↓            ↓
    Ingestion    Retrieval    Generation
        │            │            │
     Parsing      Embedding       LLM
     Chunking     Vector DB      Prompt
     Metadata     Reranking      Output
```

**If you understand these three layers, you understand the core of a RAG application.**